# Download das bibliotecas

In [10]:
!pip install pyswarm

# Importacao dos dados

In [11]:
import pandas as pd
import os
import numpy as np
import random

In [12]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
caminho_dados = '/content/drive/Shareddrives/IC - Otimizacao computacional para inflacao/Dados'

# Listar os arquivos dentro da pasta "Dados"
arquivos_dados = os.listdir(caminho_dados)

print(arquivos_dados)


['dados_macro.ipynb', 'df_macro.csv']


In [14]:
# Caminho para o arquivo CSV
caminho_arquivo = os.path.join(caminho_dados, 'df_macro.csv')

# Ler o arquivo CSV
df = pd.read_csv(caminho_arquivo)

In [15]:
df = df.set_index('Date')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 154 entries, 2012-03-01 to 2024-12-01
Data columns (total 28 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   IPCA                                   154 non-null    float64
 1   IPCA_Transportes                       154 non-null    float64
 2   IPCA Alimentação_bebidas               154 non-null    float64
 3   INPC_Habitação                         154 non-null    float64
 4   IPCA_Saúde_cuidados_pessoais           154 non-null    float64
 5   IPCA_Vestuário                         154 non-null    float64
 6   IPCA_Educação                          154 non-null    float64
 7   IPCA_Despesas_Pessoais                 154 non-null    float64
 8   USDBRL                                 154 non-null    float64
 9   Reservas Internacionais                154 non-null    int64  
 10  Desemprego                             154 non-null    float64


# Importacao das ferramentas para implementacao do PSO

In [16]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from pyswarm import pso  # Biblioteca para PSO

# Funcao fitness

In [17]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# 📌 Função de fitness (não foi alterada)
def feature_selection_fitness(mask, X_train_scaled, y_train, X_test_scaled, y_test, model, metric_func):
    """
    Função de avaliação para PSO com modelo passado como argumento e métrica.
    """
    mask = np.round(mask).astype(int)  # Converter para 0 ou 1
    selected_features = X_train_scaled.columns[mask == 1]  # Selecionar as features

    if len(selected_features) == 0:  # Caso não tenha selecionado nenhuma feature
        return 9999  # Penalização por não selecionar nenhuma feature

    # Treinar o modelo com as features selecionadas
    model.fit(X_train_scaled[selected_features], y_train)

    # Fazer previsões
    y_pred = model.predict(X_test_scaled[selected_features])

    # Calcular a métrica
    score = metric_func(y_test, y_pred)

    return score  # Retorna a métrica calculada (MAE, MSE, etc.)

# Divisao dos dados e normalizacao dos dados

In [18]:

# 📌 Preparação dos dados
X = df.drop(columns=['IPCA'])  # Removendo IPCA das features
y = df['IPCA']

# Divisão entre treino e teste (SÉRIE TEMPORAL, sem shuffle)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

Normalizacao dos dados(media 0 e desvio padrao 1)

In [19]:
# 📌 Normalização dos dados
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)


# Implementacao usando lasso na funcao fitness

In [20]:
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error
from pyswarm import pso
import numpy as np
import pandas as pd

# 📌 Garantir que X_train_scaled e X_test_scaled são DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# 📌 Função de fitness com Lasso e restrição de até 5 features
def lasso_feature_selection_fitness(mask, X_train, y_train, X_test, y_test, model, metric):
    mask = np.round(mask).astype(int)

    num_selected = mask.sum()
    if num_selected > 5 or num_selected == 0:
        return 1e6  # Penalização alta se regra for violada

    X_train_selected = X_train.iloc[:, mask == 1]
    X_test_selected = X_test.iloc[:, mask == 1]

    model.fit(X_train_selected, y_train)
    y_pred = model.predict(X_test_selected)

    return metric(y_test, y_pred)  # Lembre-se: MAE menor = melhor

# 📌 Parâmetros do PSO
num_features = X_train_scaled.shape[1]
lb = [0] * num_features  # 0 = feature desligada
ub = [1] * num_features  # 1 = feature ligada

# 📌 Modelo de regressão
lasso_model = Lasso(alpha=0.1, random_state=42)

# 📌 Executar PSO
best_mask, _ = pso(
    lambda mask: lasso_feature_selection_fitness(
        mask,
        X_train_scaled, y_train,
        X_test_scaled, y_test,
        lasso_model,
        mean_absolute_error
    ),
    lb, ub,
    swarmsize=10,
    maxiter=20,
    debug=True
)

# 📌 Arredondar e garantir que no máximo 5 features sejam selecionadas
best_mask = np.round(best_mask).astype(int)

if best_mask.sum() > 5:
    indices = np.argsort(-best_mask)[:5]  # Top 5 "mais ativadas"
    new_mask = np.zeros_like(best_mask)
    new_mask[indices] = 1
    best_mask = new_mask

# 📌 Resultado final
selected_features = X_train_scaled.columns[best_mask == 1]
print("🔹 Melhor conjunto de features selecionado pelo PSO (máx. 5):", list(selected_features))


No constraints given.
Best after iteration 1: [0.20529002 0.79121967 0.57649382 0.72046054 0.21450732 0.1665138
 0.76915699 0.9775028  0.4276912  0.02857199 0.99367729 0.97300141
 0.64416006 0.88793801 0.46901099 0.99250774 0.82068119 0.9838332
 0.31394822 0.12089856 0.87780477 0.13628374 0.84217203 0.12761526
 0.39292274 0.21857112 0.62527423] 1000000.0
Best after iteration 2: [0.20529002 0.79121967 0.57649382 0.72046054 0.21450732 0.1665138
 0.76915699 0.9775028  0.4276912  0.02857199 0.99367729 0.97300141
 0.64416006 0.88793801 0.46901099 0.99250774 0.82068119 0.9838332
 0.31394822 0.12089856 0.87780477 0.13628374 0.84217203 0.12761526
 0.39292274 0.21857112 0.62527423] 1000000.0
Best after iteration 3: [0.20529002 0.79121967 0.57649382 0.72046054 0.21450732 0.1665138
 0.76915699 0.9775028  0.4276912  0.02857199 0.99367729 0.97300141
 0.64416006 0.88793801 0.46901099 0.99250774 0.82068119 0.9838332
 0.31394822 0.12089856 0.87780477 0.13628374 0.84217203 0.12761526
 0.39292274 0.2185